# Delta Lake Basics — 04: UPDATE, DELETE, MERGE

## What you will learn
Delta's DML operations (UPDATE, DELETE, MERGE) are what make it a true
data lakehouse — not just an immutable append-only store.

In this notebook:
1. `UPDATE` — modify rows matching a condition
2. `DELETE` — remove rows matching a condition
3. `MERGE INTO` — upsert: insert + update + delete in one operation
4. How DML works internally — Copy-on-Write vs Deletion Vectors
5. MERGE patterns — CDC, SCD Type 2, deduplication
6. Performance tips for DML operations


In [ ]:
import os, time, pathlib, shutil, random, datetime, subprocess, glob
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
from delta.tables import DeltaTable

MASTER   = 'spark://spark-master:7077'
DATA_DIR = '/workspace/data/delta_basics'
pathlib.Path(DATA_DIR).mkdir(parents=True, exist_ok=True)

spark = (
    SparkSession.builder
    .appName('delta-basics')
    .master(MASTER)
    .config('spark.executor.memory', '2g')
    .config('spark.driver.memory',   '1g')
    .config('spark.executor.cores',  '2')
    .config('spark.shuffle.sort.bypassMergeThreshold', '0')
    .getOrCreate()
)
spark.sparkContext.setLogLevel('WARN')
print(f"Spark {spark.version} | Delta Lake ready")

random.seed(42)
N = 100_000
CATEGORIES = ["Electronics","Clothing","Books","Food","Sports","Furniture"]
COUNTRIES  = ["US","UK","DE","FR","JP","CA","AU","BR"]
STATUSES   = ["pending","confirmed","shipped","delivered","cancelled"]

schema = StructType([
    StructField("order_id",    LongType()),
    StructField("customer_id", LongType()),
    StructField("product",     StringType()),
    StructField("category",    StringType()),
    StructField("country",     StringType()),
    StructField("quantity",    IntegerType()),
    StructField("price",       DoubleType()),
    StructField("revenue",     DoubleType()),
    StructField("status",      StringType()),
    StructField("order_date",  DateType()),
])

base = datetime.date(2024, 1, 1)
rows = []
for i in range(N):
    qty   = random.randint(1, 10)
    price = round(random.uniform(5, 1000), 2)
    d     = base + datetime.timedelta(days=random.randint(0, 364))
    rows.append((i+1, random.randint(1,10000),
                 f"Product_{random.randint(1,200)}",
                 random.choice(CATEGORIES), random.choice(COUNTRIES),
                 qty, price, round(qty*price, 2),
                 random.choice(STATUSES), d))

df = spark.createDataFrame(rows, schema)
df.cache().count()
print(f"Dataset ready: {N:,} rows")

In [ ]:
TABLE = f"{DATA_DIR}/04_dml"
shutil.rmtree(TABLE, ignore_errors=True)
df.write.format("delta").mode("overwrite") \
        .option("delta.enableDeletionVectors", "true") \
        .save(TABLE)
initial = spark.read.format("delta").load(TABLE).count()
print(f"Table created: {initial:,} rows")
print()
print("Status distribution:")
spark.read.format("delta").load(TABLE) \
     .groupBy("status").count().orderBy(F.desc("count")).show()

## Step 1 — UPDATE

UPDATE modifies rows in-place. Delta reads affected files,
applies the update, and writes new files (Copy-on-Write).


In [ ]:
# Simple UPDATE — change status
print("Before UPDATE:")
before = spark.read.format("delta").load(TABLE).filter(F.col("status")=="pending").count()
print(f"  pending orders: {before:,}")

spark.sql(f"""
    UPDATE delta.`{TABLE}`
    SET status = 'confirmed'
    WHERE status = 'pending' AND order_id % 3 = 0
""")

after = spark.read.format("delta").load(TABLE).filter(F.col("status")=="confirmed").count()
print(f"After UPDATE (1/3 of pending → confirmed):")
print(f"  confirmed orders: {after:,}")

# Update with expression
spark.sql(f"""
    UPDATE delta.`{TABLE}`
    SET revenue = price * quantity * 0.9
    WHERE status = 'confirmed' AND country = 'US'
""")
print("Updated revenue for US confirmed orders (10% discount applied)")

DeltaTable.forPath(spark, TABLE).history() \
    .select("version","operation","operationMetrics").show(5, truncate=False)

## Step 2 — DELETE


In [ ]:
# Count before
cancelled_before = spark.read.format("delta").load(TABLE) \
                       .filter(F.col("status")=="cancelled").count()
print(f"Before DELETE — cancelled: {cancelled_before:,}")

# DELETE all cancelled orders older than June
spark.sql(f"""
    DELETE FROM delta.`{TABLE}`
    WHERE status = 'cancelled'
    AND   order_date < '2024-06-01'
""")

cancelled_after = spark.read.format("delta").load(TABLE) \
                      .filter(F.col("status")=="cancelled").count()
total_after = spark.read.format("delta").load(TABLE).count()

print(f"After DELETE — cancelled: {cancelled_after:,}  (pre-June removed)")
print(f"Total rows after DELETE : {total_after:,}")
print()

# Bulk DELETE — remove returned low-value orders
spark.sql(f"""
    DELETE FROM delta.`{TABLE}`
    WHERE price < 20 AND quantity = 1
""")
print(f"After second DELETE: {spark.read.format('delta').load(TABLE).count():,} rows")

## Step 3 — MERGE INTO: The Most Powerful DML

MERGE handles insert + update + delete atomically.
The classic use case is CDC (Change Data Capture) sync.


In [ ]:
# Simulate CDC changes arriving from source system
changes_data = []

# 500 existing orders with updated status
import random
random.seed(99)
existing_ids = random.sample(range(1, 100_001), 500)
for oid in existing_ids[:300]:
    changes_data.append((oid, "shipped", datetime.date(2024, 6, 15)))
for oid in existing_ids[300:]:
    changes_data.append((oid, "delivered", datetime.date(2024, 6, 15)))

# 200 new orders
for i in range(200):
    changes_data.append((200_001 + i, "pending", datetime.date(2024, 7, 1)))

changes_schema = StructType([
    StructField("order_id",   LongType()),
    StructField("new_status", StringType()),
    StructField("updated_at", DateType()),
])
changes_df = spark.createDataFrame(changes_data, changes_schema)
changes_df.createOrReplaceTempView("changes")

print(f"Incoming CDC changes: {changes_df.count()} rows")
print(f"  Updates : {len(existing_ids)}")
print(f"  Inserts : 200")
print()

before_count = spark.read.format("delta").load(TABLE).count()
print(f"Table before MERGE: {before_count:,} rows")

In [ ]:
# MERGE INTO — upsert pattern
spark.sql(f"""
    MERGE INTO delta.`{TABLE}` t
    USING changes s ON t.order_id = s.order_id
    WHEN MATCHED THEN
        UPDATE SET t.status = s.new_status
    WHEN NOT MATCHED THEN
        INSERT (order_id, customer_id, product, category, country,
                quantity, price, revenue, status, order_date)
        VALUES (s.order_id, 0, 'New Product', 'Unknown', 'US',
                1, 0.0, 0.0, s.new_status, s.updated_at)
""")

after_count = spark.read.format("delta").load(TABLE).count()
print(f"Table after MERGE : {after_count:,} rows  (+200 inserted)")
print()
print("MERGE metrics:")
DeltaTable.forPath(spark, TABLE).history(1) \
    .select("version","operation","operationMetrics").show(truncate=False)

## Step 4 — MERGE Patterns

MERGE is extremely flexible — here are the most useful patterns.


In [ ]:
# Pattern 1: MERGE with DELETE clause (full CDC)
spark.sql(f"""
    MERGE INTO delta.`{TABLE}` t
    USING changes s ON t.order_id = s.order_id
    WHEN MATCHED AND s.new_status = 'cancelled' THEN DELETE
    WHEN MATCHED THEN UPDATE SET t.status = s.new_status
    WHEN NOT MATCHED THEN INSERT *
""")
print("Pattern 1: MERGE with DELETE clause ✅")

# Pattern 2: INSERT-only merge (deduplication)
DEDUP_TABLE = f"{DATA_DIR}/04_dedup"
shutil.rmtree(DEDUP_TABLE, ignore_errors=True)
df.limit(10000).write.format("delta").mode("overwrite").save(DEDUP_TABLE)

dupes = df.limit(1000)  # overlap with first 1000 rows
dupes.createOrReplaceTempView("new_data")

DeltaTable.forPath(spark, DEDUP_TABLE).alias("t") \
    .merge(dupes.alias("s"), "t.order_id = s.order_id") \
    .whenNotMatchedInsertAll() \
    .execute()
print(f"\nPattern 2: INSERT-only merge (dedup) ✅")
print(f"  Before: 10,000 | After: {spark.read.format('delta').load(DEDUP_TABLE).count():,} (no dupes added)")

## Summary: DML Quick Reference

```python
# UPDATE
spark.sql("UPDATE delta.`/path` SET col = val WHERE condition")

# DELETE
spark.sql("DELETE FROM delta.`/path` WHERE condition")

# MERGE (Python API — more type-safe)
DeltaTable.forPath(spark, path).alias("target") \
    .merge(source_df.alias("source"), "target.id = source.id") \
    .whenMatchedUpdate(set={"status": "source.status"}) \
    .whenNotMatchedInsertAll() \
    .execute()

# MERGE (SQL)
spark.sql("""
    MERGE INTO delta.`path` t
    USING source s ON t.id = s.id
    WHEN MATCHED THEN UPDATE SET t.col = s.col
    WHEN NOT MATCHED THEN INSERT *
    WHEN NOT MATCHED BY SOURCE THEN DELETE  -- Spark 3.4+
""")
```

### DML performance tips
- `replaceWhere` is faster than UPDATE for full partition replacement
- Add `delta.enableDeletionVectors=true` for fast row-level deletes
- MERGE is most efficient when source is small relative to target
- Filter source before MERGE to reduce matched rows
